# Lab 1 - OpenAI Agents SDK Fundamentals

A hands-on reference for the core building blocks of the **OpenAI Agents SDK**:

| Part | Topic | Key APIs |
|------|-------|----------|
| 1 | Creating & running agents, tracing, streaming | `Agent`, `Runner.run()`, `trace()`, `Runner.run_streamed()` |
| 2 | Giving agents tools (Pushover notifications) | `@function_tool`, `tools=[...]` |
| 3 | Conversation memory | `result.to_input_list()`, `SQLiteSession` |

**Prerequisites** (loaded from `.env` via `load_dotenv()`):
- `OPENAI_API_KEY` - used implicitly by the SDK
- `PUSHOVER_USER` / `PUSHOVER_TOKEN` - for the push-notification tool in Part 2

**Key mental model:** An `Agent` is just configuration (name + instructions + model + tools). The `Runner` executes the agent loop: call the model -> run any requested tools -> feed results back -> repeat until a final answer. By default each `Runner.run()` call is **stateless** - memory must be added explicitly (Part 3).

In [3]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent # Streaming token deltas
from agents import (
    Agent, # Define an LLM agent(name + instructions + model +[tools])
    Runner, # Execute agents: await Runner.run(...)or Runner.run_streamed(...) 
    trace, # Trace agent execution steps
    function_tool, # Define tools that agents can use
    SQLiteSession, # Manage SQLite sessions for agents
)

load_dotenv()

# Package note: Install with `pip install openai-agents` or `uv add openai-agents`.
# `pip install agents` installs a different (unrelated) library.


True

## Part 1: Creating and Running an Agent

Define a minimal `Agent`, execute it with `Runner.run()`, inspect the `RunResult`, then add observability (`trace`) and token streaming (`Runner.run_streamed`).

In [4]:
# Minimal Agent: three required-ish knobs.
# 1. Name: label for traces / debugging (not sent as the system prompt).
# 2. Instructions: system prompt - defines persona and behavior.
# 3. Model: OpenAI model id string(uses OPENAI_API_KEY from the environment).
agent = Agent(
    name='Jokester',
    instructions="You're a funny AI agent. Your job is to tell jokes.",
    model='gpt-5.4-mini'
)

In [5]:
# Runner.run(agent, input) starts the agent loop and returns a RunResult.
# Must await - notebooks support top-level await in async-capable kernels.
result = await Runner.run(agent, 'Tell me a joke about Autonomous AI Agents')

In [6]:
# Final output = the agent's last message content(string here; can be structured later).
print(result.final_output)

Why did the autonomous AI agent bring a ladder to work?

Because it heard the task was “high-level autonomy.”


In [7]:
# Full turn history as a list of message dicts (user + assistant).
# Useful for debugging and for manually continuing a conversation (Part 3).
result.to_input_list()

[{'content': 'Tell me a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_0eed9df25acb4090006ae4d21a664487d1a770343d900e15c5',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent bring a ladder to work?\n\nBecause it heard the task was “high-level autonomy.”',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

### Adding Observability with Trace

In [8]:
# Same joke run, but nested under a named trace for the dashboard.
# Everything inside the `with` block shares that workflow / span name.
with trace('Telling a joke'):
    result = await Runner.run(agent, 'Tell me a joke about Autonomous AI Agents')
print(result.final_output)

Why did the autonomous AI agent break up with the coffee machine?

Because every time it said, “I’ve got this,” it immediately started auto-brewing its own problems.


### Streaming Responses

In [10]:
# Streaming variant of Runner.run- Do Not await the call itself; await events in the loop.
result = Runner.run_streamed(agent, 
                             input='Please tell me 5 jokes about AI Agents')
async for event in result.stream_events():
    # Only print character/token deltas from the model text stream.
    if event.type == 'raw_response_event' and isinstance(event.data,ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 5 AI agent jokes for you:

1. **Why did the AI agent break up with the chatbot?**  
   Because it felt like the relationship had too many unresolved intents.

2. **What do AI agents say when they’re surprised?**  
   “Well, that was outside my training data!”

3. **Why was the AI agent always calm?**  
   Because it had great emotional *model*ing.

4. **Why did the AI agent get promoted?**  
   It had excellent task execution and zero coffee breaks.

5. **What’s an AI agent’s favorite kind of music?**  
   Prompt and bass.

If you want, I can make them:
- more nerdy,
- more ridiculous,
- or more specifically about ChatGPT-style agents.

## Part 2: Adding Tools

In [18]:
# Pushover credentials from .env (loaded at the top of the notebook).
# Typical formats: user keys start with "u", app tokens start with "a".

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = 'https://api.pushover.net/1/messages.json' 

if pushover_user:
    if pushover_user.startswith('u'):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but does not look like a valid user key")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith('a'):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but does not start with 'a'")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [21]:
# Plain Python helper- NOT yet an agent tool. Call this yourself for smoke test.
def push(message):
    print(f"Push: {message}")
    if not pushover_user or not pushover_token:
        raise ValueError('PUSHOVER_USER and PUSHOVER_TOKEN must be set in the environment.')

    payload = {
        'user': pushover_user,
        'token': pushover_token,
        'message': message
    }
    response = requests.post(pushover_url, data=payload, timeout=10)
    response.raise_for_status()

In [22]:
# Smoke test for push function
push("HEY!!!")

Push: HEY!!!


In [24]:
# Still a normal function object - compare with @function_tool version below
push


<function __main__.push(message)>

In [23]:
# @function_tool turns this into a FunctionTool the LLM can call.
# The docstring is the tool description shown to the model - keep it clear and action-oriented.
# Type hints(message: str) become the JSON schema for arguments.

@function_tool
def push_tool(message: str) -> str:
    """ 
    Send the given message to the user as a push notification.
    """
    if not pushover_user or not pushover_token:
        raise ValueError('PUSHOVER_USER and PUSHOVER_TOKEN must be set in the environment.')

    payload = {
        'user': pushover_user,
        'token': pushover_token,
        'message': message
    }
    # Return a short status string so the model knows whether the call succeeded.
    response = requests.post(pushover_url, data=payload, timeout=10)
    response.raise_for_status()
    return f"Push sent with API status code {response.status_code}"

In [25]:
# After decoration, push_tool is a FunctionTool (not a plain function).
# Inspect name, description, params_json_schema, etc.
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification.', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1166d3920>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [26]:
# Argument JSON schema auto-generated from the type hints - this is exactly what the model sees.
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [27]:
# The tool description comes from the function's docstring.
push_tool.description

'Send the given message to the user as a push notification.'

In [28]:
# Agent with a tool: pass FunctionTool objects via tools=[...].
# The model decides when (and whether) to call push_tool based on its name/description/schema.
notifier = Agent(
    name='Notifier',
    model='gpt-5.4-mini',
    instructions='You notify the user upon request',
    tools=[push_tool]
)

In [29]:
# The agent autonomously calls push_tool, then reports back in final_output.
# The trace groups the LLM call + tool call under one workflow in the dashboard.
with trace('Pizza has arrived'):
    result = await Runner.run(notifier, 'Notify the user that the pizza is here')
print(result.final_output)

Done.


## Part 3 Sessions(memory)

In [30]:
# Fresh assistant agent - no tools, no session yet. Demonstrates default (no memory).
agent = Agent(
    name='Assistant',
    model='gpt-5.4-mini'
)

In [31]:
# Turn 1: introduce a name. This history is only inside `response` for this call.
response = await Runner.run(agent, 'Hi there. My name is Michael')

In [32]:
# Another independent run (also introduces the name) - still not linked to later turns.
response = await Runner.run(agent, 'Hi there. My name is Michael')
print(response.final_output)

Hi Michael! How can I assist you today?


In [33]:
# New run with NO prior history -> agent cannot know the name . Proves default is stateless.
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't have access to personal data unless you've shared it with me in this conversation. If you'd like me to know your name, feel free to tell me!


### Memory Approach 1: Manually Pass The Message List

In [34]:
# Start a fresh conversation we will continue manually
response = await Runner.run(agent, "Hello my name is Michael.")
print(response.final_output)

Hi Michael! How can I assist you today?


In [35]:
# Conversation so far as OpenAI-style message dicts (user + assistant turns).
response.to_input_list()

[{'content': 'Hello my name is Michael.', 'role': 'user'},
 {'id': 'msg_01ea7618b599d14c006ae4e3c2276887d182fbf2e3bd238740',
  'content': [{'annotations': [],
    'text': 'Hi Michael! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [36]:
# Append the next user message to the prior history list.
next_input = response.to_input_list() + [
    {
        'role': 'user',
        'content': 'What is my name?'
    }
]

In [37]:
# Pass the full history list as the input - the agent now "remembers" the name.
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Michael. How can I help you today, Michael?


### Memory Approach 2: Built in SQLiteSession

In [38]:
# Session id '123456' scopes this conversation thread.
# In-memory by default. For on-disk persistence:
# session = SQLiteSession("123456", "memory.db")
session = SQLiteSession("123456")

In [39]:
# Passing session=... stores this turn in SQLite under the session id automatically.
response = await Runner.run(
    agent,
    "Hi there. My name is Michael.",
    session=session,
)
print(response.final_output)

Hi Michael! How can I assist you today?


In [40]:
# Same session - prior turns are loaded automatically. Agent should recall "Michael".
response = await Runner.run(
    agent,
    "What's my name?",
    session=session
)
print(response.final_output)

Your name is Michael. How can I help you today?


## Conclusion & Quick Reference

**What this lab covered:**
1. **Agents are config, Runner is execution** - `Agent(name, instructions, model, tools)` defines behavior; `await Runner.run(agent, input)` executes the loop and returns a `RunResult` whose `.final_output` is the answer.
2. **Tracing** - wrap runs in `with trace('workflow name'):` to group LLM calls and tool calls into one named workflow on the OpenAI traces dashboard.
3. **Streaming** - `Runner.run_streamed()` is *not* awaited; instead `async for event in result.stream_events()` and filter for `raw_response_event` + `ResponseTextDeltaEvent` to print token deltas.
4. **Tools** - `@function_tool` converts a typed Python function into a `FunctionTool`: the docstring becomes the tool description, type hints become the argument JSON schema, and the return string tells the model what happened. Attach with `tools=[...]`.
5. **Memory** - runs are stateless by default. Two options:
   - *Manual*: `response.to_input_list() + [new_user_message]` passed as the next input.
   - *Automatic*: `SQLiteSession(session_id)` passed as `session=` - history is stored/loaded for you (in-memory by default; pass a filename for on-disk persistence).

**Gotchas to remember:**
- Install **`openai-agents`** - `pip install agents` is an unrelated package.
- Notebooks allow top-level `await`; in plain scripts use `asyncio.run(...)`.
- Keep tool docstrings action-oriented - the model relies on them to decide when to call the tool.
- A `SQLiteSession` with no filename lives only as long as the process; use `SQLiteSession(id, "memory.db")` to persist across restarts.